# Release results

This notebook reads saved metric JSON and optionally archived predictions. It does not fit models or acquire data. The frozen headline and post-freeze diagnostics remain separate.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
root = Path.cwd()
if not (root / "outputs/metrics").exists():
    root = root.parent
metrics = root / "outputs/metrics"
def read(name):
    return json.loads((metrics / name).read_text())
s = read("forecast_benchmark.json")
d = read("operator_diagnostics.json")
q = read("quality_audit.json")["authorities"]
pd.DataFrame([{k:s[k] for k in ["candidate", "g", "eligible_days", "matched_hours", "intended_hours"]}]).assign(upper95=s["upper95"][str(s["block_days"])] )

In [ ]:
table = pd.DataFrame(s["authority_metrics"])
by = table.pivot(index="balancing_authority", columns="method", values="daily_mae_mw")
by[[s["candidate"], "authority"]].assign(ratio=by[s["candidate"]]/by["authority"])

## Post-freeze operator bias

Forecast minus adjusted demand, grouped by Pacific delivery start hour. This pattern does not establish a valid demand correction.

In [ ]:
ciso = next(r for r in d["diagnostics"] if r["authority"]=="CISO" and r["target"]=="adjusted" and r["shift_hours"]==0)
hourly = pd.DataFrame(ciso["by_local_hour"])
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(hourly.hour, hourly.bias_mw, color="#0072B2")
ax.axhline(0, color="gray", linewidth=1)
ax.set(xlabel="Pacific delivery start hour", ylabel="Operator bias (MW)")
plt.show()

In [ ]:
queue = pd.DataFrame({ba:row["queue_counts"] for ba,row in q.items()}, index=["Reported", "Adjusted, original peak", "Adjusted, revised peak"]).T
queue

In [ ]:
saved = root / ".artifacts/benchmark/predictions.parquet"
if saved.exists():
    predictions = pd.read_parquet(saved)
    display(predictions.groupby("method").size().rename("saved forecast rows"))
else:
    print("Archived predictions are optional for these metric-based reproductions.")